# GenCare AI
---
Auteur:   Eva Rombouts  
Datum:    26-12-2023.  
Doel:     Ervaring opdoen met Python, NLP  
Script:   Onderzoekt een aantal opensource modellen 

Data gegenereerd met de OpenAI API.  
---

In [2]:
#from datasets import load_dataset, Dataset, DatasetDict
#from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
#import torch
#import time
#import evaluate
import pandas as pd
import numpy as np
#from peft import LoraConfig, get_peft_model, TaskType
#from sklearn.model_selection import train_test_split

In [3]:
# # Wanneer in colab
# from google.colab import drive
# drive.mount('/content/drive')
# gci = pd.read_csv('/content/drive/MyDrive/eColab/GenCareAI/data/gci_rapportages.csv')
# gci.info()

In [4]:
gci = pd.read_csv('../zorgdata/df_rapportages.csv')
gci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6341 entries, 0 to 6340
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   afdeling     6341 non-null   object 
 1   client_id    6341 non-null   object 
 2   weekno       6341 non-null   int64  
 3   dag          6341 non-null   object 
 4   niveau       6341 non-null   object 
 5   rapportage   6341 non-null   object 
 6   onrustscore  6334 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 346.9+ KB


In [5]:
gci['onrustscore'].describe()

count    6334.000000
mean       36.650931
std        21.987785
min         0.000000
25%        15.000000
50%        35.000000
75%        55.000000
max        95.000000
Name: onrustscore, dtype: float64

In [6]:
gci['rapportage'].head()

0    Meneer Koster begint de dag rustig. Ik heb hem...
1    Vandaag was meneer Koster wat onrustig in de o...
2    Meneer Koster was deze ochtend wat vergeetacht...
3    Meneer Koster begon de dag rustig en was meega...
4    Vandaag was meneer Koster wat onrustig en verw...
Name: rapportage, dtype: object

In [7]:
# Om een gevoel te krijgen van de lengte van de rapportages:
gci['rapportage_lengte'] = gci['rapportage'].apply(len)
gci['n_woorden'] = gci['rapportage'].str.split().apply(len)
gci[['rapportage_lengte','n_woorden']].describe()

,rapportage_lengte,n_woorden
count,6341.000000,6341.000000
mean,526.779530,89.428639
std,125.737814,22.123274
min,211.000000,34.000000
25%,442.000000,74.000000
50%,512.000000,87.000000
75%,594.000000,101.000000
max,1133.000000,200.000000


## Modellen zoeken

In [8]:
from transformers import pipeline

In [9]:
model = 'wietsedv/bert-base-dutch-cased-finetuned-sentiment'
sentiment_analysis = pipeline('sentiment-analysis', model = model)

In [14]:
# Laten we starten met de eerste week van dhr Verhoeven
test_zinnen = gci.loc[(gci['client_id'] == 'kamer02') & (gci['afdeling'] == 'eikenboom') & (gci['weekno'] == 1), 'rapportage'].tolist()
for rp in test_zinnen:
    print(rp)

Vandaag heeft meneer Verhoeven goed meegewerkt tijdens de ochtendzorg. Hij kon zichzelf aankleden en wassen, maar had hulp nodig bij het aantrekken van zijn sokken en schoenen vanwege zijn rugpijn. We hebben extra ondersteuning geboden bij het douchen, gezien zijn eerdere duizeligheid. Meneer Verhoeven was redelijk rustig en helder vandaag en heeft genoten van een potje schaken met een medebewoner. Hij had wat moeite om zich te concentreren, maar genoot desondanks van het spel. We hebben hem eraan herinnerd dat hij altijd bij ons terecht kan als hij zich verward voelt.
Vandaag heeft meneer Verhoeven zichzelf grotendeels kunnen redden tijdens de ochtendzorg. Hij moest wel af en toe herinnerd worden aan de volgorde van handelingen, maar dit verliep over het algemeen soepel. Tijdens het schaken met een medebewoner, raakte meneer Verhoeven af en toe gefrustreerd. Hij kon zich niet goed concentreren en maakte verschillende fouten. Dit leidde tot lichte onrust en wat boosheid. We hebben gepr

In [16]:
gci.loc[(gci['client_id'] == 'kamer02') & (gci['afdeling'] == 'eikenboom') & (gci['weekno'] == 1), 'onrustscore']

70    30.0
71    40.0
72    15.0
73    55.0
74    10.0
75    50.0
76     5.0
Name: onrustscore, dtype: float64

In [15]:
se_output = sentiment_analysis(test_zinnen)

for a in enumerate(se_output) :
    print(a)

(0, {'label': 'pos', 'score': 0.999996542930603})
(1, {'label': 'neg', 'score': 0.9998700618743896})
(2, {'label': 'pos', 'score': 0.9999958276748657})
(3, {'label': 'neg', 'score': 0.9999489784240723})
(4, {'label': 'pos', 'score': 0.9999964237213135})
(5, {'label': 'neg', 'score': 0.9999390840530396})
(6, {'label': 'pos', 'score': 0.9999966621398926})


Ik ben niet erg onder de indruk... Hij neigt erg naar positiviteit. 
Laten we er nog een proberen

In [17]:
model="henryk/bert-base-multilingual-cased-finetuned-dutch-squad2"

qa_pipeline = pipeline(
    "question-answering",
    model = model,
    tokenizer = model
)

Some weights of the model checkpoint at henryk/bert-base-multilingual-cased-finetuned-dutch-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [20]:
for rp in test_zinnen:
    qa_resultaat = qa_pipeline({
        'context': rp,
#        'question': "Hoe heet deze client?"})
#        'question': "Wat heeft de client nodig?"})
        'question': "Hoe voelt deze client zich?"})
    print(rp)
    print(qa_resultaat['answer'])
    print('-'*100)


Vandaag heeft meneer Verhoeven goed meegewerkt tijdens de ochtendzorg. Hij kon zichzelf aankleden en wassen, maar had hulp nodig bij het aantrekken van zijn sokken en schoenen vanwege zijn rugpijn. We hebben extra ondersteuning geboden bij het douchen, gezien zijn eerdere duizeligheid. Meneer Verhoeven was redelijk rustig en helder vandaag en heeft genoten van een potje schaken met een medebewoner. Hij had wat moeite om zich te concentreren, maar genoot desondanks van het spel. We hebben hem eraan herinnerd dat hij altijd bij ons terecht kan als hij zich verward voelt.
verward
----------------------------------------------------------------------------------------------------
Vandaag heeft meneer Verhoeven zichzelf grotendeels kunnen redden tijdens de ochtendzorg. Hij moest wel af en toe herinnerd worden aan de volgorde van handelingen, maar dit verliep over het algemeen soepel. Tijdens het schaken met een medebewoner, raakte meneer Verhoeven af en toe gefrustreerd. Hij kon zich niet g

In [21]:
sum_model = 'yhavinga/t5-v1.1-base-dutch-cnn-test'

In [22]:
# Maak de summarization pipeline
summarization_pipeline = pipeline(
    "summarization",
    model=sum_model,
    tokenizer=sum_model
)

In [23]:
# Voer de samenvatting uit
samenvatting_resultaat = summarization_pipeline(test_zinnen[1])

In [24]:
samenvatting = samenvatting_resultaat[0]['summary_text']
print(samenvatting)

Vandaag heeft meneer Verhoeven zichzelf grotendeels kunnen redden tijdens de ochtendzorg. Hij kon zich niet goed concentreren en maakte verschillende fouten. Dit leidde tot lichte onrust tijdens het schaken met een medebewoner. We hebben geprobeerd hem gerust te stellen, maar dit verliep over het algemeen soepel. Meneer Verhoeven is nu weer terug bij de rust.


In [25]:
for rp in test_zinnen:
    sum_resultaat = summarization_pipeline(rp, min_length = 50, max_length=70)
    smry = sum_resultaat[0]['summary_text']
    print(rp)
    print(smry)
    print('-'*100)


Vandaag heeft meneer Verhoeven goed meegewerkt tijdens de ochtendzorg. Hij kon zichzelf aankleden en wassen, maar had hulp nodig bij het aantrekken van zijn sokken en schoenen vanwege zijn rugpijn. We hebben extra ondersteuning geboden bij het douchen, gezien zijn eerdere duizeligheid. Meneer Verhoeven was redelijk rustig en helder vandaag en heeft genoten van een potje schaken met een medebewoner. Hij had wat moeite om zich te concentreren, maar genoot desondanks van het spel. We hebben hem eraan herinnerd dat hij altijd bij ons terecht kan als hij zich verward voelt.
Meneer Verhoeven heeft genoten van een potje schaken met een medebewoner. Hij had hulp nodig bij het aantrekken van zijn sokken en schoenen vanwege zijn eerdere duizeligheid. We hebben hem eraan herinnerd dat hij altijd bij ons terecht kan als hij zich verward voelt.
----------------------------------------------------------------------------------------------------
Vandaag heeft meneer Verhoeven zichzelf grotendeels kun

Wow, dat werkt best goed. 